# AI-Model-Atlas 🗺️ | Cognitive RAG Quickstart

Welcome! This notebook lets you play with the **AI-Model-Atlas Cognitive RAG system** in 30 seconds with **zero local setup**.

We will run a simulated instance of our cognitive pipeline showcasing:
- 🔄 **Query Rewriting**
- ⚡ **Semantic Cache Hit (0.00s)**
- 🎯 **Relevance Reranking (Margin filter)**
- 🛡️ **Execution Controller (Retry & Failover)**

### 1. Install & Import Dependencies

In [ ]:
!pip install numpy sentence-transformers
import time
import numpy as np

### 2. Define Cognitive RAG Simulator
We implement a self-contained version of our pipeline directly in this notebook so you can see the log telemetry in action.

In [ ]:
class CognitiveRAG:
    def __init__(self):
        self.cache = {}
        # Simple knowledge base
        self.kb = {
            "llama 3 license parameters": "Llama 3 is licensed under the LLAMA 3 Community License Agreement, free for commercial use under 700M monthly active users.",
            "semantic cache optimization": "Semantic Cache maps vector embeddings to stored answers using cosine similarity distance thresholds, avoiding LLM calls."
        }
        
    def rewrite_query(self, query):
        # Simple normalizer
        clean = query.lower().strip("?!. ").replace("tell me about ", "")
        print(f"[🔄 Rewrite] Normalized Intent: '{clean}'")
        return clean
        
    def check_cache(self, query):
        # Simulate semantic lookup
        if query in self.cache:
            print(f"[⚡ Cache] Hit! ✅ Retrieval bypassed.")
            return self.cache[query]
        print(f"[⚡ Cache] Miss! ❌ Routing to retrieval.")
        return None
        
    def retrieve_and_rerank(self, query):
        print(f"[🎯 Retrieve] Searching ChromaDB vector store...")
        time.sleep(0.5)
        
        # Look for match in KB
        best_match = None
        best_score = 0.0
        
        for k, v in self.kb.items():
            # Mock cosine similarity score
            overlap = len(set(query.split()) & set(k.split())) / max(len(query.split()), 1)
            if overlap > best_score:
                best_score = overlap
                best_match = v
                
        if best_match and best_score > 0.4:
            print(f"[🎯 Rerank] Passed Margin Filter! (Score: {best_score:.2f})")
            return best_match
        print(f"[🎯 Rerank] No highly relevant document found. (Best Score: {best_score:.2f})")
        return "Fallback: System failed to retrieve relevant documents."
        
    def execute(self, query):
        print(f"\n--- Processing Query: '{query}' ---")
        start_time = time.time()
        
        # 1. Rewrite
        normalized = self.rewrite_query(query)
        
        # 2. Cache
        cached = self.check_cache(normalized)
        if cached:
            elapsed = time.time() - start_time
            print(f"[Response]: {cached} (Latency: {elapsed:.4f}s)")
            return
            
        # 3. Retrieve & Rerank
        doc = self.retrieve_and_rerank(normalized)
        
        # 4. LLM Generation Simulation
        print(f"[🛡️ Controller] Routing query to Hybrid LLM API...")
        time.sleep(0.4)
        response = f"[LLM Response generated from retrieved context]: {doc}"
        
        # Populate cache
        self.cache[normalized] = response
        
        elapsed = time.time() - start_time
        print(f"[Response]: {response} (Latency: {elapsed:.4f}s)")

### 3. Run RAG Pipeline Test Drive
Watch the first query miss the cache and retrieve context, and the second query hit the cache for sub-millisecond execution!

In [ ]:
rag = CognitiveRAG()

# First Run: Cache Miss
rag.execute("Tell me about Llama 3 License parameters?")

print("\n" + "="*40 + "\n")

# Second Run: Cache Hit!
rag.execute("Tell me about Llama 3 License parameters?")